## 1. Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path("data")
TABLES_DIR = Path("thesis_tables")
TABLES_DIR.mkdir(exist_ok=True)

FRAMEWORK_ORDER = ["Unity", "Xcode"]
AR_MODE_ORDER   = ["Markerbased", "Markerless"]
SESSION_ORDER   = ["2 min", "5 min"]
GROUP_COLS = ["framework", "ar_mode", "session"]

pd.set_option("display.precision", 1)
pd.set_option("display.width", 160)

## 2. Load and sanity-check data

In [ ]:
df = pd.read_csv(DATA_DIR / "run_summary.csv")

df["framework"] = pd.Categorical(df["framework"], FRAMEWORK_ORDER, ordered=True)
df["ar_mode"]   = pd.Categorical(df["ar_mode"],   AR_MODE_ORDER,   ordered=True)
df["session"]   = pd.Categorical(df["session"],   SESSION_ORDER,   ordered=True)

print(f"Total runs: {len(df)}")
print(f"\nRuns per condition:")
df.groupby(GROUP_COLS, observed=True).size().reset_index(name="n")

In [ ]:
def by_condition(spec):
    return (
        df.groupby(GROUP_COLS, observed=True)
          .agg(**{k: v for k, v in spec.items()})
          .round(1)
          .reset_index()
          .rename(columns={"framework": "Plattform",
                           "ar_mode":   "AR Type",
                           "session":   "Session"})
    )

def save_table(table, name, title, caption, label):
    md = f"**{title}**\n\n*{caption}*\n\n" + table.to_markdown(index=False)
    (TABLES_DIR / f"{name}.md").write_text(md)
    latex = table.to_latex(index=False, escape=True,
                           column_format="l" * len(table.columns))
    wrapped = (f"\\begin{{table}}[h]\n  \\centering\n"
               f"  \\caption{{{title}. {caption}}}\n"
               f"  \\label{{tab:{label}}}\n{latex}\\end{{table}}\n")
    (TABLES_DIR / f"{name}.tex").write_text(wrapped)
    print(f"  wrote {name}.md and {name}.tex")

## 3. Results tables

### Table 1 — Frame rate

In [ ]:
fps_table = by_condition({
    "Mean FPS":                ("fps_mean", "mean"),
    "Run-to-run Variation":    ("fps_mean", "std"),
    "Worst-case FPS":          ("fps_p5", "mean"),
    "Frames below target (%)": ("fps_frames_below_55_pct", "mean"),
})
save_table(fps_table)
fps_table

### Table 2 — Thermal state

In [ ]:
thermal_table = by_condition({
    "Time Nominal (%)":  ("pct_nominal", "mean"),
    "Time Fair (%)":     ("pct_fair", "mean"),
    "Time Serious (%)":  ("pct_serious", "mean"),
})
save_table(thermal_table)
thermal_table

### Table 3 — Hangs

In [ ]:
hangs_table = by_condition({
    "Hang events":          ("hang_count",        "mean"),
    "Total hang time (ms)": ("duration_total_ms", "mean"),
    "Longest hang (ms)":    ("duration_max_ms",   "max"),
})
save_table(hangs_table)
hangs_table

### Table 4 — CPU and threads

In [ ]:
cpu_table = by_condition({
    "Mean CPU (%)":   ("cpu_percent_mean",   "mean"),
    "Peak CPU (%)":   ("cpu_percent_max",    "mean"),
    "Active threads": ("thread_count_mean",  "mean"),
})
cpu_table["Active threads"] = cpu_table["Active threads"].round(0).astype(int)
save_table(cpu_table)
cpu_table

### Table 5 — Memory

In [ ]:
memory_table = by_condition({
    "Peak memory (MB)":     ("memory_footprint_peak_mb", "mean"),
    "Peak app heap (MB)":   ("memory_anonymous_mb_max",  "mean"),
    "Memory growth (MB)":   ("memory_growth_mb",         "mean"),
})
for c in ["Peak memory (MB)", "Peak app heap (MB)", "Memory growth (MB)"]:
    memory_table[c] = memory_table[c].round(0).astype(int)
save_table(memory_table)
memory_table

### Table 6 — GPU

In [ ]:
gpu_table = by_condition({
    "Mean GPU usage (%)":   ("gpu_device_util_mean",     "mean"),
    "Mean GPU memory (MB)": ("gpu_in_use_sys_mem_mean",  "mean"),
})
for c in ["Mean GPU usage (%)", "Mean GPU memory (MB)"]:
    gpu_table[c] = gpu_table[c].round(0).astype(int)
save_table(gpu_table)
gpu_table

## 4. Visualizations

In [ ]:
import numpy as np

FIGURES_DIR = Path("thesis_figures")
FIGURES_DIR.mkdir(exist_ok=True)

In [ ]:
plt.rcParams.update({
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})

COLOR_UNITY = "#1f77b4"
COLOR_XCODE = "#ff7f0e"

In [ ]:
def save_fig(fig, name):
    pdf_path = FIGURES_DIR / f"{name}.pdf"
    png_path = FIGURES_DIR / f"{name}.png"
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, bbox_inches="tight")
    print(f"  saved {pdf_path} and {png_path}")

def condition_label(framework, ar_mode, session):
    fw_short = "U" if framework == "Unity" else "X"
    ar_short = "MB" if ar_mode == "Markerbased" else "ML"
    return f"{fw_short}-{ar_short}\n{session}"

In [ ]:
thermal_means = (df.groupby(GROUP_COLS, observed=True)
                   [["pct_nominal", "pct_fair", "pct_serious", "pct_critical"]]
                   .mean()
                   .reset_index())
thermal_means["_label"] = thermal_means.apply(
    lambda r: condition_label(r["framework"], r["ar_mode"], r["session"]), axis=1
)

fig, ax = plt.subplots(figsize=(9, 4))
bottom = np.zeros(len(thermal_means))
state_colors = {
    "pct_nominal":  ("Nominal",  "#4caf50"),
    "pct_fair":     ("Fair",     "#ffc107"),
    "pct_serious":  ("Serious",  "#ff7043"),
    "pct_critical": ("Critical", "#b71c1c"),
}
for col, (label, color) in state_colors.items():
    values = thermal_means[col].values
    ax.bar(thermal_means["_label"], values, bottom=bottom, label=label, color=color)
    bottom += values

ax.set_ylabel("% of run time")
ax.set_xlabel("Testing Condition")
ax.set_ylim(0, 100)
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
save_fig(fig, "fig_thermal_distribution")
plt.show()

In [ ]:
fps_samples = pd.read_csv(DATA_DIR / "samples_fps.csv")

condition_lookup = df.set_index("trace")[["framework", "ar_mode", "session"]]
fps_samples = fps_samples.join(condition_lookup, on="trace")

BIN_SIZE_S = 10
fps_samples["time_bin"] = (fps_samples["timestamp_s"] // BIN_SIZE_S) * BIN_SIZE_S

binned = (fps_samples.groupby(GROUP_COLS + ["time_bin"], observed=True)["fps"]
          .mean().reset_index())

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, session in zip(axes, ["2 min", "5 min"]):
    subset = binned[binned["session"] == session]
    for (fw, ar), group in subset.groupby(["framework", "ar_mode"], observed=True):
        color = COLOR_UNITY if fw == "Unity" else COLOR_XCODE
        linestyle = "-" if ar == "Markerbased" else "--"
        ax.plot(group["time_bin"], group["fps"],
                label=f"{fw} {ar}", color=color, linestyle=linestyle, linewidth=1.5)
    ax.set_xlabel("Time since session start (s)")
    ax.set_title(f"{session} sessions")
    ax.axhline(60, color="grey", linestyle=":", linewidth=0.8, alpha=0.7)
    ax.axhline(30, color="grey", linestyle=":", linewidth=0.8, alpha=0.7)
    ax.set_ylim(20, 65)
    ax.grid(alpha=0.3)

axes[0].set_ylabel("Frame rate (FPS)")
axes[1].legend(loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False)
plt.tight_layout()
save_fig(fig, "fig_fps_over_time")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

for fw in FRAMEWORK_ORDER:
    for ar in AR_MODE_ORDER:
        subset = df[(df["framework"] == fw) & (df["ar_mode"] == ar)]
        color = COLOR_UNITY if fw == "Unity" else COLOR_XCODE
        marker = "o" if ar == "Markerbased" else "^"
        ax.scatter(subset["gpu_device_util_mean"], subset["pct_serious"],
                   color=color, marker=marker, s=60, alpha=0.7,
                   edgecolors="white", linewidth=0.8,
                   label=f"{fw} {ar}")

ax.set_xlabel("Mean GPU utilization (%)")
ax.set_ylabel("Time in Serious thermal state (%)")
ax.legend(loc="upper left", frameon=False)
ax.grid(alpha=0.3)
plt.tight_layout()
save_fig(fig, "fig_gpu_vs_thermal")
plt.show()

In [ ]:
metrics_for_corr = {
    "Mean FPS":       "fps_mean",
    "P5 FPS":         "fps_p5",
    "Mean CPU":       "cpu_percent_mean",
    "Mean GPU":       "gpu_device_util_mean",
    "Peak memory":    "memory_footprint_peak_mb",
    "Time Serious":   "pct_serious",
    "Hang count":     "hang_count",
}

corr_df = df[list(metrics_for_corr.values())].copy()
corr_df.columns = list(metrics_for_corr.keys())
corr = corr_df.corr(method="pearson")

fig, ax = plt.subplots(figsize=(7, 5.5))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.index)

for i in range(len(corr.index)):
    for j in range(len(corr.columns)):
        value = corr.iloc[i, j]
        text_color = "white" if abs(value) > 0.5 else "black"
        ax.text(j, i, f"{value:.2f}", ha="center", va="center",
                color=text_color, fontsize=9)

cbar = fig.colorbar(im, ax=ax, label="Pearson correlation", shrink=0.8)
plt.tight_layout()
save_fig(fig, "fig_correlation_heatmap")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), sharey=True)

for ax, fw in zip(axes, FRAMEWORK_ORDER):
    subset = df[df["framework"] == fw][list(metrics_for_corr.values())].copy()
    subset.columns = list(metrics_for_corr.keys())
    fw_corr = subset.corr(method="pearson")

    im = ax.imshow(fw_corr, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(range(len(fw_corr.columns)))
    ax.set_yticks(range(len(fw_corr.index)))
    ax.set_xticklabels(fw_corr.columns, rotation=45, ha="right")
    ax.set_yticklabels(fw_corr.index)
    ax.set_title(f"{fw}  (n = {len(subset)} runs)")

    for i in range(len(fw_corr.index)):
        for j in range(len(fw_corr.columns)):
            value = fw_corr.iloc[i, j]
            text_color = "white" if abs(value) > 0.5 else "black"
            ax.text(j, i, f"{value:.2f}", ha="center", va="center",
                    color=text_color, fontsize=8)

fig.colorbar(im, ax=axes, label="Pearson correlation",
             shrink=0.7, pad=0.02)
save_fig(fig, "fig_correlation_by_framework")
plt.show()

## 5. Direct access

In [ ]:
(df.sort_values("fps_mean")
   .groupby(GROUP_COLS, observed=True)
   .head(1)[["trace", "run", "fps_mean", "fps_p5"]])

In [ ]:
(df.query("framework == 'Unity' and ar_mode == 'Markerbased' and session == '2 min'")
   [["run", "fps_mean", "fps_p5", "cpu_percent_mean", "memory_footprint_peak_mb"]]
   .sort_values("fps_mean"))